# Mini LLM — Kaggle GPU Edition

LLM dari nol dengan teknik **parameter sharing** (ALBERT-style):
- 1 set bobot dipakai N kali untuk depth virtual N
- Ukuran file tetap kecil meski depth tinggi
- Tanpa quantize, tanpa 4-bit, tanpa kompresi

**Cara pakai:**
1. Settings → Accelerator → GPU T4 x2 (atau P100)
2. Settings → Internet → On (untuk pip install kalau perlu)
3. Run All

In [ ]:
# Cell 1: Setup & cek GPU
import subprocess, sys
try:
    import torch
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'torch', 'numpy', '-q'])
    import torch

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    DEVICE = 'cuda'
else:
    print('WARNING: GPU gak ketemu. Pakai CPU.')
    DEVICE = 'cpu'

import numpy as np
print('Setup done. Device =', DEVICE)

In [ ]:
# Cell 2: Konfigurasi model
# Versi sandbox: d_model=96, layers=12 (750 KB)
# Versi Kaggle GPU — bisa lebih gede!

CFG = {
    'vocab_size': 1024,     # 3x sandbox
    'd_model':    256,      # ~2.7x sandbox
    'n_heads':    8,
    'd_ff':       1024,     # 4x d_model
    'num_layers': 24,       # virtual depth (gak nambah size!)
    'max_seq_len': 256,
}
EPOCHS = 2000
BATCH = 64
LR = 3e-4
SEED = 42

# Estimasi size sebelum training
d, v = CFG['d_model'], CFG['vocab_size']
f = CFG['d_ff']; n = CFG['max_seq_len']
uniq = v*d + n*d + 4*d*d + 3*d*f + 3*d  # emb + attn + mlp + norm
print(f'Estimasi unique params: {uniq:,}')
print(f'Estimasi file size:    {uniq*4/1024/1024:.2f} MB (fp32)')
print(f'Virtual depth:           {CFG["num_layers"]}')
print(f'Effective (if no share): {uniq * CFG["num_layers"]:,}')

In [ ]:
# Cell 3: Tokenizer BPE (dari nol, tanpa library eksternal)
import re, json
from collections import Counter
from pathlib import Path

PAT_SIMPLE = r'\s+|\w+|\S'

def _get_pairs(word):
    pairs = set()
    prev = word[0]
    for ch in word[1:]:
        pairs.add((prev, ch))
        prev = ch
    return pairs

class MiniBPETokenizer:
    def __init__(self):
        self.merges = {}
        self.vocab = {}
        self.pad_id, self.bos_id, self.eos_id = 0, 1, 2
        
    def train(self, text, target_vocab=1024, verbose=True):
        words_raw = re.findall(PAT_SIMPLE, text)
        word_freqs = Counter(words_raw)
        words = {w: list(w.encode('utf-8')) for w in word_freqs}
        self.vocab = {i: bytes([i]) for i in range(256)}
        self.merges = {}
        num_merges = target_vocab - 256
        for i in range(num_merges):
            pair_counts = Counter()
            for w, syms in words.items():
                for p in _get_pairs(syms):
                    pair_counts[p] += word_freqs[w]
            if not pair_counts: break
            best, _ = pair_counts.most_common(1)[0]
            new_id = 256 + i
            self.merges[best] = new_id
            self.vocab[new_id] = self.vocab[best[0]] + self.vocab[best[1]]
            new_words = {}
            for w, syms in words.items():
                new_words[w] = self._merge_word(syms, best, new_id)
            words = new_words
            if verbose and (i % 100 == 0 or i == num_merges-1):
                print(f'  [bpe] merge {i+1}/{num_merges}')
        self.vocab_size = max(self.vocab.keys()) + 1
        print(f'[bpe] done. vocab_size={self.vocab_size}')
    
    @staticmethod
    def _merge_word(syms, pair, new_id):
        new = []; i = 0
        while i < len(syms):
            if i < len(syms)-1 and (syms[i], syms[i+1]) == pair:
                new.append(new_id); i += 2
            else:
                new.append(syms[i]); i += 1
        return new
    
    def encode(self, text):
        ids = [self.bos_id]
        for w in re.findall(PAT_SIMPLE, text):
            syms = list(w.encode('utf-8'))
            while len(syms) >= 2:
                pairs = _get_pairs(syms)
                best = None; best_rank = None
                for p in pairs:
                    if p in self.merges:
                        rank = list(self.merges.keys()).index(p)
                        if best_rank is None or rank < best_rank:
                            best = p; best_rank = rank
                if best is None: break
                syms = self._merge_word(syms, best, self.merges[best])
            ids.extend(syms)
        ids.append(self.eos_id)
        return ids
    
    def decode(self, ids):
        out = b''
        for i in ids:
            if i in (self.pad_id, self.bos_id, self.eos_id): continue
            if i in self.vocab: out += self.vocab[i]
        return out.decode('utf-8', errors='replace')
    
    def save(self, path):
        Path(path).write_text(json.dumps({
            'merges': [[list(k), v] for k, v in self.merges.items()],
            'vocab_size': self.vocab_size,
            'pad_id': self.pad_id, 'bos_id': self.bos_id, 'eos_id': self.eos_id,
        }, ensure_ascii=False))
    
    @classmethod
    def load(cls, path):
        d = json.loads(Path(path).read_text())
        tok = cls()
        tok.merges = {tuple(p): v for p, v in d['merges']}
        tok.vocab = {i: bytes([i]) for i in range(256)}
        for pair, new_id in sorted(tok.merges.items(), key=lambda x: x[1]):
            tok.vocab[new_id] = tok.vocab[pair[0]] + tok.vocab[pair[1]]
        tok.vocab_size = d['vocab_size']
        tok.pad_id = d['pad_id']; tok.bos_id = d['bos_id']; tok.eos_id = d['eos_id']
        return tok

print('Tokenizer class loaded.')

In [ ]:
# Cell 4: Model MiniGPT (dari nol, dengan parameter sharing)
import math
import torch.nn as nn
import torch.nn.functional as F

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, attn_bias=True):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.qkv = nn.Linear(d_model, 3 * d_model, bias=attn_bias)
        self.proj = nn.Linear(d_model, d_model, bias=attn_bias)
        self.scale = 1.0 / math.sqrt(self.d_head)
    def forward(self, x, kv_cache=None):
        B, T, C = x.shape
        qkv = self.qkv(x)
        q, k, v = qkv.split(C, dim=-1)
        q = q.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        if kv_cache is not None:
            kp, vp = kv_cache
            if kp is not None:
                k = torch.cat([kp, k], dim=2)
                v = torch.cat([vp, v], dim=2)
            new_cache = (k, v)
        else:
            new_cache = None
        T_full = k.size(2)
        causal = torch.tril(torch.ones(T, T_full, device=x.device, dtype=torch.bool))
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.masked_fill(~causal.unsqueeze(0).unsqueeze(0), float('-inf'))
        attn = F.softmax(attn, dim=-1)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(out), new_cache

class SwiGLUMLP(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.w1 = nn.Linear(d_model, d_ff, bias=False)
        self.w2 = nn.Linear(d_ff, d_model, bias=False)
        self.w3 = nn.Linear(d_model, d_ff, bias=False)
    def forward(self, x):
        return self.w2(F.silu(self.w1(x)) * self.w3(x))

class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
        self.eps = eps
    def forward(self, x):
        var = x.pow(2).mean(-1, keepdim=True)
        return x * torch.rsqrt(var + self.eps) * self.weight

class SharedBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = RMSNorm(cfg['d_model'])
        self.attn = CausalSelfAttention(cfg['d_model'], cfg['n_heads'])
        self.norm2 = RMSNorm(cfg['d_model'])
        self.mlp = SwiGLUMLP(cfg['d_model'], cfg.get('d_ff'))
    def forward(self, x, kv_cache=None):
        a, nc = self.attn(self.norm1(x), kv_cache=kv_cache)
        x = x + a
        x = x + self.mlp(self.norm2(x))
        return x, nc

class MiniGPT(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg['vocab_size'], cfg['d_model'])
        self.pos_emb = nn.Embedding(cfg['max_seq_len'], cfg['d_model'])
        self.block = SharedBlock(cfg)
        self.norm_f = RMSNorm(cfg['d_model'])
        self.head = nn.Linear(cfg['d_model'], cfg['vocab_size'], bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying
        self.apply(self._init_weights)
    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None: torch.nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            torch.nn.init.normal_(m.weight, mean=0.0, std=0.02)
    def forward(self, idx, targets=None, kv_caches=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        new_caches = []
        for i in range(self.cfg['num_layers']):
            cache = kv_caches[i] if kv_caches else None
            x, nc = self.block(x, kv_cache=cache)
            new_caches.append(nc)
        x = self.norm_f(x)
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, self.cfg['vocab_size']),
                                   targets.view(-1), ignore_index=-100)
        return logits, loss, new_caches
    @torch.no_grad()
    def generate(self, idx, max_new_tokens=50, temperature=1.0, top_k=None, eos_id=None):
        self.eval()
        kv_caches = [None] * self.cfg['num_layers']
        for t in range(max_new_tokens):
            idx_cond = idx if t == 0 else idx[:, -1:]
            logits, _, kv_caches = self.forward(idx_cond, kv_caches=kv_caches)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('inf')
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_id], dim=1)
            if eos_id is not None and next_id.item() == eos_id: break
        return idx
    def save(self, path):
        torch.save({'state_dict': self.state_dict(), 'cfg': self.cfg}, path)
    @classmethod
    def load(cls, path, map_location='cpu'):
        ckpt = torch.load(path, map_location=map_location, weights_only=False)
        m = cls(ckpt['cfg']); m.load_state_dict(ckpt['state_dict']); return m
    def count_params(self):
        return sum(p.numel() for p in self.parameters())

print('Model class loaded.')

In [ ]:
# Cell 5: Dataset (Kaggle — pakai data lebih besar)
# Opsi 1: pakai teks Wikipedia Indonesia via kaggle datasets
# Opsi 2: pakai Project Gutenberg (inggris)
# Untuk demo, kita pakai teks kustom yang lebih besar dari sandbox

import os
from pathlib import Path

# Jika ada dataset kaggle (add via 'Add Input' di kanan):
# - cari 'wikipedia indonesia' dataset
# - taruh pathnya di sini
# KAGGLE_DATASET_PATH = '/kaggle/input/indonesian-wikipedia/idwiki.txt'

# Untuk starter, kita generate teks yang lebih bervariasi
# (kalau mau serius, ganti dengan dataset beneran)
corpus = '''
Indonesia adalah negara kepulauan terbesar di dunia.
Negara ini terdiri dari lebih dari tujuh ribu pulau yang tersebar di khatulistiwa.
Ibu kota Indonesia adalah Jakarta, kota yang penuh dengan sejarah.
Bahasa resmi negara Indonesia adalah bahasa Indonesia.
Mata uang resmi negara Indonesia adalah rupiah.
Penduduk Indonesia sangat beragam, terdiri dari ratusan suku bangsa.
Setiap suku memiliki budaya, bahasa, dan tradisi yang unik.
Indonesia kaya akan sumber daya alam, mulai dari minyak hingga emas.
Hutan tropis Indonesia termasuk yang terbesar di dunia.
Laut Indonesia menyimpan kekayaan ikan dan terumbu karang.
Gunung berapi banyak terdapat di Indonesia karena ring of fire.
Iklim Indonesia adalah iklim tropis dengan dua musim.
Musim hujan dan musim kemarau silih berganti sepanjang tahun.
Pertanian menjadi mata pencaharian utama bagi banyak penduduk.
Padi adalah tanaman pangan utama yang ditanam di sawah.
Nasi merupakan makanan pokok bagi sebagian besar masyarakat Indonesia.
Indonesia memiliki sejarah panjang yang penuh dengan perjuangan.
Proklamasi kemerdekaan dibacakan pada tanggal tujuh belas agustus.
Pahlawan nasional Indonesia banyak jumlahnya dari berbagai daerah.
Pendidikan adalah kunci untuk membangun masa depan bangsa.
Teknologi semakin berkembang pesat di era modern ini.
Komputer dan internet mengubah cara manusia berkomunikasi.
Kecerdasan buatan atau artificial intelligence menjadi tren global.
Model bahasa besar atau large language model sedang populer.
Transformer adalah arsitektur dasar dari model bahasa modern.
Attention mechanism memungkinkan model memahami konteks kalimat.
Pelatihan model bahasa membutuhkan data yang sangat besar.
Komputasi yang dibutuhkan juga sangat tinggi dan mahal.
Peneliti terus berusaha membuat model yang lebih efisien.
Parameter sharing adalah teknik untuk mengurangi ukuran model.
Satu set bobot dapat digunakan berulang kali lintas lapisan.
ALBERT adalah contoh model yang menggunakan parameter sharing.
Dengan teknik ini, ukuran file tetap kecil meski kedalaman tinggi.
Inovasi seperti ini penting untuk efisiensi komputasi.
Masa depan kecerdasan buatan sangat menjanjikan.
Manusia dan mesin akan bekerja sama secara harmonis.
Pengetahuan adalah kekuatan yang tak ternilai harganya.
Belajar setiap hari membuat hidup menjadi lebih bermakna.
Kesuksesan datang kepada mereka yang tidak pernah menyerah.
Kerja keras dan ketekunan adalah kunci pencapaian.
Indonesia akan terus maju dan berkembang seiring waktu.
Anak muda adalah harapan masa depan bangsa Indonesia.
Mari kita bangun negeri ini bersama-sama dengan semangat pantang menyerah.
'''
Path('corpus.txt').write_text(corpus)
print(f'Corpus size: {len(corpus)} chars, {len(corpus.split())} words')

In [ ]:
# Cell 6: Train tokenizer
tok = MiniBPETokenizer()
tok.train(corpus, target_vocab=CFG['vocab_size'], verbose=True)
tok.save('tokenizer.json')
print('Tokenizer saved.')

# Sanity test
enc = tok.encode('Indonesia adalah negara')
print('encoded:', enc)
print('decoded:', tok.decode(enc))

In [ ]:
# Cell 7: Build dataset
import torch
from torch.utils.data import Dataset, DataLoader

class LangDataset(Dataset):
    def __init__(self, text, tokenizer, seq_len):
        ids = tokenizer.encode(text)
        self.samples = []
        seq_len_plus = seq_len + 1
        for i in range(0, len(ids) - seq_len_plus, seq_len_plus // 2):  # overlapping window
            chunk = ids[i:i+seq_len_plus]
            if len(chunk) < seq_len_plus:
                chunk = chunk + [tokenizer.pad_id] * (seq_len_plus - len(chunk))
            x = chunk[:-1]
            y = [t if t != tokenizer.pad_id else -100 for t in chunk[1:]]
            self.samples.append((x, y))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        x, y = self.samples[i]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

ds = LangDataset(corpus, tok, CFG['max_seq_len'])
print(f'Dataset size: {len(ds)} samples')
dl = DataLoader(ds, batch_size=BATCH, shuffle=True, drop_last=False)

In [ ]:
# Cell 8: Init model & cek info
torch.manual_seed(SEED)
model = MiniGPT(CFG).to(DEVICE)
n_params = model.count_params()
print(f'Unique params:  {n_params:,}  ({n_params*4/1024/1024:.2f} MB @ fp32)')
print(f'Virtual depth:  {CFG["num_layers"]}')
print(f'Effective:      {n_params * CFG["num_layers"]:,}  (seakan tanpa sharing)')

In [ ]:
# Cell 9: Training loop
import time
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)

t0 = time.time()
losses = []
for epoch in range(EPOCHS):
    model.train()
    total = 0.0; n = 0
    for x, y in dl:
        x, y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad()
        _, loss, _ = model(x, targets=y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item(); n += 1
    scheduler.step()
    avg = total / max(n, 1)
    losses.append(avg)
    if epoch % 50 == 0 or epoch == EPOCHS - 1:
        elapsed = time.time() - t0
        print(f'epoch {epoch+1:4d}/{EPOCHS}  loss={avg:.4f}  t={elapsed:.1f}s')

print(f'\nTraining done in {time.time()-t0:.1f}s')

In [ ]:
# Cell 10: Save model & tokenizer
import os
os.makedirs('output', exist_ok=True)
model.save('output/mini_llm.pt')
Path('output/tokenizer.json').write_text(Path('tokenizer.json').read_text())
size_kb = os.path.getsize('output/mini_llm.pt') / 1024
print(f'Model saved:    output/mini_llm.pt ({size_kb:.1f} KB)')
print(f'Tokenizer saved: output/tokenizer.json')

In [ ]:
# Cell 11: Generate text
model_cpu = model.cpu()  # generate di CPU biar portable
model_cpu.eval()

def generate(prompt, max_new=50, temperature=0.8, top_k=20):
    ids = tok.encode(prompt)
    idx = torch.tensor([ids], dtype=torch.long)
    out = model_cpu.generate(idx, max_new_tokens=max_new, temperature=temperature,
                              top_k=top_k, eos_id=tok.eos_id)
    return tok.decode(out[0].tolist())

for prompt in ['Indonesia', 'Belajar', 'Teknologi', 'Masa depan']:
    print(f'\n[{prompt}] ->')
    print(generate(prompt))

## Next Steps

1. **Ganti dataset** dengan corpus lebih besar (1MB+). Coba dataset:
   - Wikipedia Indonesia (cari di kaggle datasets)
   - Berita Indonesia
   - Buku Project Gutenberg

2. **Scale up CFG** (gratis kalau cuma naik num_layers!):
   ```python
   CFG['num_layers'] = 48  # virtual depth 48, file tetap sama
   CFG['d_model'] = 512   # ini baru nambah size 4x
   ```

3. **Train lebih lama**: set EPOCHS = 5000 atau sampai loss < 2.5

4. **Download output**: klik tab `output/` di kanan, download `mini_llm.pt` dan `tokenizer.json`